# VSCS per-drive perception on Colab (P3-T2, P4-T3, P4-T4)

Detector (RT-DETR, Apache-2.0, ADR 0011) + perception: static objects into the persistent height map, people/vehicles/cyclists into the tracker. Writes `obstacles.jsonl`.

> **No logic here** (CLAUDE.md §4.3): install, mount, call `scripts/perceive.py`.

**Before running:**
- Upload an extracted lot-run frames folder (`scripts/extract_frames.py`) to Drive.
- The camera must be calibrated (P0-T7) and its mount measured (`configs/capture.yaml mount`), committed and pushed; the script refuses otherwise.
- Without ego poses the van is assumed **stationary** (right for the P4-T4 walk-behind clip).

**Budget** (§1): T4. Log runtime and compute units in the devlog (R-11).

## 1. GPU check

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. Mount Drive and point at the frames

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

VSCS_DRIVE = Path("/content/drive/MyDrive/vscs")
# Edit to the frames run you uploaded (the folder holding frames.jsonl):
FRAMES_RUN = VSCS_DRIVE / "data/processed/capture/<YYYYMMDD-HHMM_sha>"
OUT_ROOT = VSCS_DRIVE / "data/processed"
print(FRAMES_RUN, "OK" if (FRAMES_RUN / "frames.jsonl").exists() else "MISSING")

## 3. Get the code and install

Colab's own torch is kept: nothing here installs or pins torch (R-12). `hf_xet` is needed for the weights download (ADR 0007).

In [ ]:
import os

REPO_URL = "https://github.com/AlvinKalaivannan/VSCS-Vehicle-Scanning-Collision-System-.git"
REPO_DIR = Path("/content/vscs")
if not REPO_DIR.exists():
    !git clone --branch main {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
!pip install --quiet pyyaml "pydantic>=2" numpy scipy shapely opencv-python transformers hf_xet pillow

## 4. Run perception

In [ ]:
!python scripts/perceive.py --frames-run "{FRAMES_RUN}" --device cuda --out-root "{OUT_ROOT}"

## 5. Before closing the tab

- [ ] The perception run folder (`obstacles.jsonl`, `detections.jsonl`, `run.log`) is on **Drive**.
- [ ] Runtime and compute units noted in today's devlog (R-11).
- [ ] Any detector misses on poles/kerbs noted against R-07.